# セグメンテーションデータセット バリデーション — 実行用ノートブック

`README.md` の「3. 手順」をセルに分けて実行できるようにしたもの。
コマンドの意味・出力の読み方・設計判断の理由は README 本体に書いてあるので、
ここでは繰り返さない。**このノートブックはコマンドを打つ手間を減らすためのもの**であり、
判断の根拠が必要になったら都度 README を参照すること。

上から順に実行すれば README と同じ1周ができる。飛ばしたいセルはスキップしてよい。

対応: [README.md](../README.md) 3章 / 参照: [docs/dataset_format.md](../docs/dataset_format.md)

## 0. 前提

- カーネルはこのリポジトリの `uv` 環境（`.venv`）に紐づいていること。
  紐づいていない場合は一度ターミナルで以下を実行してからカーネルを選び直す。

  ```bash
  cd /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation
  uv sync --group notebook --group review   # jupyter + fiftyone 一式
  uv run python -m ipykernel install --user --name segmentation-validation
  ```

- 目視レビュー（3.6〜3.8）まで行うなら `--group review` を付けたままにする
  （外すと fiftyone 一式がアンインストールされる。README 2章）。
- 以降のセルは `!` から始まる**シェルコマンド**（README のコマンドをそのまま実行）と、
  結果を読むための Python セルが混在している。

In [ ]:
import os

PROJECT_ROOT = "/mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation"
os.chdir(PROJECT_ROOT)
os.environ.setdefault("FIFTYONE_DATABASE_DIR", os.path.expanduser("~/.fiftyone/var/lib/mongo"))
print("cwd:", os.getcwd())
print("FIFTYONE_DATABASE_DIR:", os.environ["FIFTYONE_DATABASE_DIR"])

### 環境確認（任意）

`mongod` のデータ置き場が NFS 上だと壊れる（README 2章）。ローカルディスクか確認する。

In [ ]:
!findmnt -T "$HOME" -o TARGET,SOURCE,FSTYPE

In [ ]:
# cv2 が1種類だけか（opencv-python と headless が共存すると壊れる）
!uv run python -c "import importlib.metadata as m, cv2; \
  print(cv2.__version__, [d.metadata['Name'] for d in m.distributions() \
  if d.metadata['Name'] and 'opencv' in d.metadata['Name'].lower()])"

## 3.1 対象を確認する

In [ ]:
!uv run segmentation-validation list-sources   # 対象JSON 3本

In [ ]:
!uv run segmentation-validation list-checks    # 登録チェック一覧

In [ ]:
!uv run segmentation-validation show-config    # 解決後の設定

## 3.2 走査（画素を読む）

**2〜3分**かかる（`--jobs 8`）。2回目以降は走査済みを飛ばして1秒未満で終わる。
共有サーバーなので、走らせる前に他ジョブが張り付いていないか確認しておく。

In [ ]:
!cat /proc/loadavg
!uptime

In [ ]:
!uv run segmentation-validation scan --jobs 8

再走査したいとき（キャッシュ破棄）や、まず少数で試したいときは以下を使う
（普段は不要。コメントアウトのまま残してある）。

```python
# !uv run segmentation-validation scan --jobs 8 --force
# !uv run segmentation-validation scan --jobs 8 --limit 20   # スモークテスト
```

## 3.3 チェックを実行する

→ `issues.json` / `issues.csv`。**error があると exit 1** を返す
（このセルは失敗しても後続を止めたくないので `!` の戻り値は無視している。
 明示的に確認したいときは次のセルの `$?` を見る）。

In [ ]:
# 2行に分けて `!` を別々に実行すると $? が引き継がれない（別プロセスになる）ため、
# 1行にまとめて exit code を拾う。
!uv run segmentation-validation check; echo "exit code: $?"

パス規約や bbox の閾値を詰めているときは、JSONだけで判定できるチェックを1秒で回せる。

> **`--only` / `--skip` を使ったら、`select` の前に `--only` なしで回し直すこと。**
> 部分実行のまま `select` すると採否が壊れる（README 3.3 参照）。

In [ ]:
# !uv run segmentation-validation check --only M06,M07

## 3.4 採否を確定する

→ `selection_decisions.csv` と `image_decisions.csv`。
初回は `review_decisions.json` が無いので、目視対象は `pending` のままになる（正しい挙動）。

In [ ]:
!uv run segmentation-validation select

## 3.5 結果を見る

`report` で `summary.md`、`gui` で `dashboard.html` を書き出す。
ノートブックなのでコマンドを打つだけでなく、生成された Markdown をそのままここに表示できる。

In [ ]:
!uv run segmentation-validation report   # -> summary.md
!uv run segmentation-validation gui      # -> dashboard.html

In [ ]:
import glob
from pathlib import Path

# fingerprint 付きディレクトリのうち最新のものを拾う
validation_dirs = sorted(
    glob.glob("output/validation/*/"), key=lambda p: Path(p).stat().st_mtime
)
latest_validation_dir = Path(validation_dirs[-1]) if validation_dirs else None
print("latest validation dir:", latest_validation_dir)

In [ ]:
from IPython.display import Markdown, display

display(Markdown((latest_validation_dir / "summary.md").read_text()))

### dashboard.html をブラウザで開く

ノートブックのセル内に埋め込む（IFrame / data URI）方法も試したが、
1240px 幅で組んである表・グラフをノートブックの狭い出力枠に押し込めると読みにくい。
**実際のブラウザタブで開いたほうが読める。** そこでこのプロセス内で軽量HTTPサーバーを
立てて、クリックできるリンクを出す。

ポートは固定（`DASHBOARD_PORT`）にしてある。README 3.6 の FiftyOne（5151）と同じ理由で、
**手元のSSHトンネル設定を1回書けば毎回使い回せる**ようにするため。
`http://localhost:{port}` を開けないときは、本サーバーへのSSH接続にポート転送が
乗っていない（`ssh -L` はSSH接続を新しく張るときにしか付けられない。VS Code Remoteの
接続や既に開いているターミナルには後から追加できない）。

**その場しのぎ**: 手元PCで新しいターミナルを開き、下のセルが出す `ssh -L ...` をそのまま
実行して接続を張ったままにする（閉じるとトンネルも切れる）。

**毎回打つのが面倒なら**、手元PCの `~/.ssh/config` に1行足す
（README 3.6 で FiftyOne 用に張っている `Host pi6` ブロックがあれば、そこに追加でよい）。

```
Host pi6
    HostName <本サーバー>
    ProxyJump <踏み台>
    LocalForward 8899 localhost:8899   # このノートブックの dashboard 用
    LocalForward 5151 localhost:5151   # FiftyOne 用（README 3.6）
```

設定はSSH接続の開始時にしか読まれないので、保存したら**今の接続を閉じて
`ssh pi6` で繋ぎ直す**こと（VS Code Remote ならウィンドウごと再接続）。

In [ ]:
import functools
import http.server
import threading

from IPython.display import Markdown, display

DASHBOARD_PORT = 8899  # SSHトンネル設定を使い回せるように固定ポートにしてある

if "dashboard_httpd" not in dir():
    dashboard_httpd = None

if dashboard_httpd is None:
    _dashboard_handler = functools.partial(
        http.server.SimpleHTTPRequestHandler, directory=str(latest_validation_dir)
    )
    dashboard_httpd = http.server.ThreadingHTTPServer(
        ("127.0.0.1", DASHBOARD_PORT), _dashboard_handler
    )
    threading.Thread(target=dashboard_httpd.serve_forever, daemon=True).start()

dashboard_url = f"http://localhost:{DASHBOARD_PORT}/dashboard.html"
display(
    Markdown(
        f"**[{dashboard_url}]({dashboard_url})** をブラウザで開く。\n\n"
        "開けないときは手元PCの新しいターミナルで以下を実行してから開く\n"
        f"（`~/.ssh/config` に `LocalForward` を書いた場合は不要）。\n\n"
        f"```bash\nssh -L {DASHBOARD_PORT}:localhost:{DASHBOARD_PORT} <本サーバー>\n```"
    )
)

In [ ]:
# 見終わったらサーバーを止める（止めなくても支障は無い。カーネルを落とせば消える）
# dashboard_httpd.shutdown()
# dashboard_httpd.server_close()
# dashboard_httpd = None

In [ ]:
import pandas as pd

selection_df = pd.read_csv(latest_validation_dir / "selection_decisions.csv")
selection_df["final_decision"].value_counts()

In [ ]:
image_df = pd.read_csv(latest_validation_dir / "image_decisions.csv")
image_df["final_decision"].value_counts()

## 3.6 FiftyOne で目視する

- `uv sync --group review` 済みであること。
- 初回は6〜9分かかる（README 3.6）。既定では「目視対象の画像」しか載らない。
  keep になった画像も全部載せたいときは `--all` を付ける（約2GB / 初回25〜30分）。

In [ ]:
!uv run segmentation-validation review build

In [ ]:
# keep になった画像も全部見たいとき（単発）。既定のままでよければ実行不要。
# !uv run segmentation-validation review build --all

### App の起動

`review launch` は `localhost:5151` でサーバーを起動し続けるコマンドなので、
**このセルで直接 `!` 実行すると kernel がブロックされ、他のセルが動かせなくなる。**
バックグラウンドプロセスとして起動し、あとで明示的に止める。

In [ ]:
import subprocess
import time

review_log = open("output/review_launch.log", "w")
review_proc = subprocess.Popen(
    ["uv", "run", "segmentation-validation", "review", "launch"],
    stdout=review_log,
    stderr=subprocess.STDOUT,
)
time.sleep(3)
print("PID:", review_proc.pid, "-> output/review_launch.log を参照")

手元の端末から SSH トンネルで開く（README 3.6）。

```bash
ssh -L 5151:localhost:5151 <本サーバー>
```

ブラウザで `http://localhost:5151` を開く。**インターネットへ公開しないこと。**

目視の判定入力は README [3.7](../README.md#37-app-での目視のやりかた) を参照
（`review_status` / `review_reason` / `reviewer` を Detection と Sample の
両方に正しく入れる、という部分だけはノートブックでは肩代わりできない）。

目視が終わったら次のセルで App を止める。

In [ ]:
review_proc.terminate()
review_proc.wait(timeout=10)
print("stopped:", review_proc.poll())

## 3.8 判定を採否へ反映する

```
review launch →（目視）→ review export → select → report
```

このループを pending が 0 になるまで繰り返す。

In [ ]:
!uv run segmentation-validation review export   # -> review_decisions.json / .csv
!uv run segmentation-validation select           # -> 採否へ反映
!uv run segmentation-validation review status    # 進捗

In [ ]:
!uv run segmentation-validation review precision  # -> precision.md

In [ ]:
display(Markdown((latest_validation_dir / "precision.md").read_text()))

### DB を作り直した後に判定を戻す（round-trip）

`review build` は DB を作り直すコマンドなので、その前に必ず `review export` して
判定を失わないようにする。作り直した後は `review import` で `review_decisions.json`
から判定を復元する。

In [ ]:
!uv run segmentation-validation review export    # ★先に必ず export
!uv run segmentation-validation review build      # DB 再構築（export 済みなら安全）
!uv run segmentation-validation review import      # review_decisions.json から判定を復元

## 3.9 開発用データセットを生成する

`pending` / `uncertain` が残っていると**既定で止まる**（exit 1）。それが正しい。

In [ ]:
!uv run segmentation-validation build-dataset; echo "exit code: $?"

先に進める必要がある場合は「許可」と「扱い」の**両方**を明示する
（`--allow-pending` だけではエラーになる）。**通常は使わない — 目視を先に終わらせること。**

In [ ]:
# from datetime import date
# tag = date.today().strftime("%Y%m%d")
# !uv run segmentation-validation build-dataset \
#   --allow-pending --pending-as exclude \
#   --version-tag {tag}

In [ ]:
development_dirs = sorted(
    glob.glob("output/development/*/*/"), key=lambda p: Path(p).stat().st_mtime
)
if development_dirs:
    latest_development_dir = Path(development_dirs[-1])
    print("latest development dir:", latest_development_dir)
    display(Markdown((latest_development_dir.parent / "selection_summary.md").read_text()))
else:
    print("development.json はまだ生成されていない")

## 3.10 単一症例の重畳図（debug 用）

FiftyOne を使わずに1症例だけ図で確認したいとき。ノートブックなら画像をそのままセル出力に表示できる。

In [ ]:
!uv run python -m segmentation_validation.overlay_masks --institution kajinoki

In [ ]:
overlay_pngs = sorted(
    glob.glob("output/overlay/*.png"), key=lambda p: Path(p).stat().st_mtime
)
overlay_pngs[-3:]

In [ ]:
from IPython.display import Image

Image(filename=overlay_pngs[-1]) if overlay_pngs else None

In [ ]:
# 症例やオリジナルマスクを指定したいとき
# !uv run python -m segmentation_validation.overlay_masks --study CXASW00000278_002 --original

## 参考: 設定を変えて回す

閾値やポリシーを変えたら `check` → `select` → `report` → `gui` を回す
（`scan` は不要。計測値は閾値に依存しない）。設定キーの一覧は README 7章。

In [ ]:
!uv run segmentation-validation show-config

In [ ]:
# 例: 気胸だけを検証する
# !uv run segmentation-validation --set 'validation.target_labels=["Findings/010"]' check
# !uv run segmentation-validation --set 'validation.target_labels=["Findings/010"]' select
# !uv run segmentation-validation --set 'validation.target_labels=["Findings/010"]' report